In [1]:
import pandas as pd

sets = pd.read_csv("lego_sets_1975_2025.csv")
themes = pd.read_csv("lego_themes_1975_2025.csv")
minifigures = pd.read_csv("lego_minifigures_1975_2025.csv")

print("Sets:", sets.shape)
print("Themes:", themes.shape)
print("Minifigures:", minifigures.shape)



Sets: (18540, 10)
Themes: (464, 4)
Minifigures: (15790, 5)


In [2]:
tables = {
    "sets": sets,
    "themes": themes,
    "minifigures": minifigures
}

for name, df in tables.items():
    print(f"\n--- {name.upper()} ---")

    print("\nTipos de datos:")
    print(df.dtypes)

    print("\nValores faltantes:")
    print(df.isna().sum())

    print("\nFilas duplicadas:")
    print(df.duplicated().sum())


--- SETS ---

Tipos de datos:
set_num                str
name                   str
year                 int64
theme_name             str
parent_theme           str
num_parts            int64
num_minifigs         int64
num_unique_colors    int64
has_minifigs          bool
img_url                str
dtype: object

Valores faltantes:
set_num              0
name                 0
year                 0
theme_name           0
parent_theme         0
num_parts            0
num_minifigs         0
num_unique_colors    0
has_minifigs         0
img_url              0
dtype: int64

Filas duplicadas:
0

--- THEMES ---

Tipos de datos:
id              int64
theme_name        str
parent_theme      str
total_sets      int64
dtype: object

Valores faltantes:
id              0
theme_name      0
parent_theme    0
total_sets      0
dtype: int64

Filas duplicadas:
0

--- MINIFIGURES ---

Tipos de datos:
fig_num                  str
name                     str
num_parts              int64
img_url        

In [3]:
for name, df in tables.items():
    print(
        name,
        "| Rows:", len(df),
        "| Missing:", df.isna().sum().sum(),
        "| Duplicates:", df.duplicated().sum()
    )

sets | Rows: 18540 | Missing: 0 | Duplicates: 0
themes | Rows: 464 | Missing: 0 | Duplicates: 0
minifigures | Rows: 15790 | Missing: 0 | Duplicates: 0


In [4]:
def clean_text_columns(df):
    cleaned = df.copy()

    text_columns = cleaned.select_dtypes(
        include=["object", "string"]
    ).columns

    for column in text_columns:
        cleaned[column] = (
            cleaned[column]
            .astype("string")
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .replace("", pd.NA)
        )

    return cleaned

In [5]:
sets_clean = clean_text_columns(sets)
themes_clean = clean_text_columns(themes)
minifigures_clean = clean_text_columns(minifigures)

print("Text cleaning completed.")

Text cleaning completed.


In [6]:
def clean_integer_columns(df, columns):
    cleaned = df.copy()

    for column in columns:
        numbers = pd.to_numeric(cleaned[column], errors="coerce")

        valid = (
            numbers.notna()
            & (numbers >= 0)
            & (numbers < 2**63)
            & (numbers % 1 == 0)
        )

        cleaned[column] = numbers.where(valid).astype("Int64")

    return cleaned

In [7]:
sets_clean = clean_integer_columns(
    sets_clean,
    ["year", "num_parts", "num_minifigs", "num_unique_colors"]
)

themes_clean = clean_integer_columns(
    themes_clean,
    ["id", "total_sets"]
)

minifigures_clean = clean_integer_columns(
    minifigures_clean,
    ["num_parts", "total_sets_appeared"]
)

for name, df in {
    "sets": sets_clean,
    "themes": themes_clean,
    "minifigures": minifigures_clean
}.items():
    print(name, "| Missing after cleaning:", df.isna().sum().sum())

sets | Missing after cleaning: 0
themes | Missing after cleaning: 0
minifigures | Missing after cleaning: 0


In [8]:
def remove_exact_duplicates(df):
    return df.drop_duplicates().reset_index(drop=True)


def validate_primary_key(df, key):
    if df[key].isna().any():
        raise ValueError(f"Missing IDs in {key}")

    if df[key].duplicated().any():
        raise ValueError(f"Conflicting duplicate IDs in {key}")

In [9]:
sets_clean = remove_exact_duplicates(sets_clean)
themes_clean = remove_exact_duplicates(themes_clean)
minifigures_clean = remove_exact_duplicates(minifigures_clean)

validate_primary_key(sets_clean, "set_num")
validate_primary_key(themes_clean, "id")
validate_primary_key(minifigures_clean, "fig_num")

print("Primary keys validated.")
print("Sets:", len(sets_clean))
print("Themes:", len(themes_clean))
print("Minifigures:", len(minifigures_clean))

Primary keys validated.
Sets: 18540
Themes: 464
Minifigures: 15790


In [10]:
theme_columns = ["theme_name", "parent_theme"]

repeated_themes = themes_clean[
    themes_clean.duplicated(
        subset=theme_columns,
        keep=False
    )
]

display(
    repeated_themes.sort_values(theme_columns)
)

,id,theme_name,parent_theme,total_sets
113,228,Creator,Seasonal,34
396,214,Creator,Seasonal,4
10,525,Technic,Educational and Dacta,280
86,529,Technic,Educational and Dacta,48


In [11]:
def create_theme_groups(df):
    groups = df[["theme_name", "parent_theme"]].copy()

    if groups.isna().any().any():
        raise ValueError("Missing theme labels")

    groups = groups.drop_duplicates().reset_index(drop=True)

    return groups

In [12]:
theme_groups = create_theme_groups(themes_clean)

print("Original themes:", len(themes_clean))
print("Unique theme groups:", len(theme_groups))

Original themes: 464
Unique theme groups: 462


In [13]:
def validate_set_themes(sets_df, groups_df):
    columns = ["theme_name", "parent_theme"]

    if sets_df[columns].isna().any().any():
        raise ValueError("Missing theme labels in sets")

    checked = sets_df.merge(
        groups_df,
        on=columns,
        how="left",
        validate="many_to_one",
        indicator=True
    )

    if (checked["_merge"] == "left_only").any():
        raise ValueError("Some sets have no matching theme group")

    return checked.drop(columns="_merge")

In [14]:
sets_clean = validate_set_themes(sets_clean, theme_groups)

print("Sets after validation:", len(sets_clean))

Sets after validation: 18540


In [15]:
def add_decade(df):
    cleaned = df.copy()
    cleaned["decade"] = (cleaned["year"] // 10) * 10
    return cleaned

In [16]:
sets_clean = add_decade(sets_clean)

display(sets_clean[["name", "year", "decade"]].head())

,name,year,decade
0,Death Star,2025,2020
1,Tropical Aquarium,2025,2020
2,River Steamboat,2025,2020
3,Adventure in Transylvania,2025,2020
4,The Thieves of Tortuga,2025,2020


In [17]:
def clean_minifig_flag(df):
    cleaned = df.copy()

    cleaned["has_minifigs"] = (
        cleaned["has_minifigs"]
        .astype("string")
        .str.strip()
        .str.lower()
        .map({"true": True, "false": False})
        .astype("boolean")
    )

    known = cleaned["num_minifigs"].notna()

    cleaned.loc[known, "has_minifigs"] = (
        cleaned.loc[known, "num_minifigs"] > 0
    )

    return cleaned

In [18]:
sets_clean = clean_minifig_flag(sets_clean)

display(
    sets_clean[["name", "num_minifigs", "has_minifigs"]].head()
)

,name,num_minifigs,has_minifigs
0,Death Star,40,True
1,Tropical Aquarium,0,False
2,River Steamboat,0,False
3,Adventure in Transylvania,15,True
4,The Thieves of Tortuga,13,True


In [19]:
def fill_missing_names(df):
    cleaned = df.copy()
    cleaned["name"] = cleaned["name"].fillna("Unknown")
    return cleaned

In [20]:
sets_clean = fill_missing_names(sets_clean)
minifigures_clean = fill_missing_names(minifigures_clean)

In [21]:
import importlib
import transform

importlib.reload(transform)

cleaned_tables = transform.transform_data(
    sets, themes, minifigures
)

for name, df in cleaned_tables.items():
    print(name, "| Rows:", len(df))

sets | Rows: 18540
themes | Rows: 464
minifigures | Rows: 15790
theme_groups | Rows: 462
